# Практическая работа №12
## Разработка специализированного LLM-агента с веб-интеграцией

**Специализация:** интеллектуальный помощник по глубокому обучению.

In [ ]:
# Сохранение моделей Ollama на Google Drive (один раз)
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

src = os.path.expanduser("~/.ollama")
dst = "/content/drive/MyDrive/ollama_models"

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Модели сохранены на Google Drive")
else:
    print("Модели уже есть на Drive")

In [2]:
%%capture

# Системная зависимость для установки Ollama
!apt-get update -qq
!apt-get install -y -qq zstd

# Python-библиотеки для LLM, RAG, PDF, Tavily и интерфейса
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-ollama \
    langchain-chroma \
    langchain-tavily \
    chromadb \
    pymupdf \
    gradio \
    ollama \
    python-dotenv

# Установка Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [3]:
import gradio
import ollama
import chromadb
import fitz

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_tavily import TavilySearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

print("Все библиотеки успешно установлены и импортированы")

Все библиотеки успешно установлены и импортированы


/tmp/ipykernel_1264/2062151442.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [14]:
import subprocess
import time
import ollama

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for attempt in range(15):
    try:
        ollama.list()
        print("Ollama успешно запущен")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Не удалось запустить Ollama")

Ollama успешно запущен


In [15]:
# # Основная LLM для ответов агента
# !ollama pull aya-expanse:8b

# # Модель эмбеддингов для PDF-RAG
# !ollama pull embeddinggemma

In [16]:
import ollama
import time

MODEL_NAME = "aya-expanse:8b"

question = "Кратко объясни, что такое переобучение нейронной сети."

start_time = time.time()

response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "Ты — интеллектуальный помощник по глубокому обучению. "
                "Отвечай только на грамотном русском языке. "
                "Не используй слова и символы из других языков."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ],
    options={"temperature": 0.1}
)

print(response["message"]["content"])
print(f"\nВремя генерации: {time.time() - start_time:.1f} сек.")

Переобучение (overfitting) нейронной сети — это ситуация, когда модель слишком хорошо адаптируется к обучающим данным, улавливая не только полезную информацию, но и случайные шумы или аномалии. В результате сеть начинает плохо обобщать новые, ранее не виденные данные. Это происходит из-за избыточной сложности модели по сравнению с объемом данных, что приводит к чрезмерному подстраиванию под конкретный набор обучающих примеров.

Время генерации: 18.4 сек.


In [17]:
# Конфигурация агента
MODEL_NAME = "aya-expanse:8b"
EMBEDDING_MODEL = "embeddinggemma"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
RETRIEVER_K = 5

print("Основная модель:", MODEL_NAME)
print("Модель эмбеддингов:", EMBEDDING_MODEL)

Основная модель: aya-expanse:8b
Модель эмбеддингов: embeddinggemma


In [20]:
import os
from google.colab import userdata
from langchain_tavily import TavilySearch

# Получение API-ключа из защищенного хранилища Colab
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

# Инструмент веб-поиска
web_search = TavilySearch(
    max_results=3,
    topic="general",
    search_depth="basic"
)

print("Tavily успешно настроен")

Tavily успешно настроен


In [22]:
def search_web(query):
    """
    Выполняет веб-поиск и подготавливает контекст для LLM.
    """
    try:
        response = web_search.invoke({"query": query})

        if "error" in response:
            return "Ошибка веб-поиска.", []

        results = response.get("results", [])

        if not results:
            return "Релевантные материалы не найдены.", []

        context_parts = []
        sources = []

        for index, item in enumerate(results, start=1):
            title = item.get("title", "Без названия")
            url = item.get("url", "")
            content = item.get("content", "")

            context_parts.append(
                f"[Источник {index}]\n"
                f"Название: {title}\n"
                f"URL: {url}\n"
                f"Фрагмент: {content}"
            )

            sources.append(f"{index}. {title}\n   {url}")

        return "\n\n".join(context_parts), sources

    except Exception as error:
        return f"Ошибка веб-поиска: {error}", []


# Проверка функции
web_context, sources = search_web(
    "Какие методы помогают бороться с переобучением нейронных сетей?"
)

print("КОНТЕКСТ ДЛЯ МОДЕЛИ:\n")
print(web_context)

print("\n\nСПИСОК ИСТОЧНИКОВ:\n")
print("\n".join(sources))

КОНТЕКСТ ДЛЯ МОДЕЛИ:

[Источник 1]
Название: Методы борьбы с переобучением искусственных нейронных сетей
URL: https://na-journal.ru/2-2019-tehnicheskie-nauki/1703-metody-borby-s-pereobucheniem-iskusstvennyh-neironnyh-setei
Фрагмент: Батч-нормализация;; Метод ансамблей;; Ранняя остановка;; Dropout. Каждый метод имеет свои достоинства и недостатки, поэтому выбор метода для борьбы с проблемой

[Источник 2]
Название: МЕТОДЫ БОРЬБЫ С ПЕРЕОБУЧЕНИЕМ В НЕЙРОННЫХ СЕТЯХ
URL: https://cyberleninka.ru/article/n/metody-borby-s-pereobucheniem-v-neyronnyh-setyah
Фрагмент: Ранняя остановка представляет метод регуляризации при обучении модели с помощью итеративного метода, похожего на градиентный спуск. Поскольку все нейронные сети

[Источник 3]
Название: Dropout - метод борьбы с переобучением нейронной сети
URL: https://proproprogs.ru/neural_network/dropout-metod-borby-s-pereobucheniem-neyronnoy-seti
Фрагмент: Узнаете принцип работы Dropout (дропаут) для борьбы с переобучением сети. Рекомендации его ис

In [23]:
import ollama
import time

SYSTEM_PROMPT = """
Ты — специализированный интеллектуальный помощник по глубокому обучению.
Отвечай только на грамотном русском языке.

Правила работы:
1. Используй предоставленный контекст веб-поиска.
2. Не выдумывай факты, которых нет в контексте.
3. Если информации недостаточно, прямо сообщи об этом.
4. Формулируй ответ понятно и структурированно.
5. Не добавляй ссылки самостоятельно: список источников будет добавлен отдельно.
"""


def answer_with_web_search(question):
    """
    Выполняет веб-поиск и формирует ответ локальной LLM.
    """
    web_context, sources = search_web(question)

    user_prompt = f"""
ВОПРОС ПОЛЬЗОВАТЕЛЯ:
{question}

КОНТЕКСТ ИЗ ВЕБ-ПОИСКА:
{web_context}

Сформируй ответ на вопрос пользователя.
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        options={"temperature": 0.1}
    )

    answer = response["message"]["content"]

    if sources:
        answer += "\n\nИсточники:\n" + "\n".join(sources)

    return answer


# Проверка работы агента
question = "Какие методы помогают бороться с переобучением нейронных сетей?"

start_time = time.time()
answer = answer_with_web_search(question)

print("Вопрос:", question)
print("\nОтвет агента:\n")
print(answer)
print(f"\nВремя выполнения: {time.time() - start_time:.1f} сек.")

Вопрос: Какие методы помогают бороться с переобучением нейронных сетей?

Ответ агента:

Для борьбы с переобучением нейронных сетей можно использовать несколько эффективных методов:

1. **Батч-нормализация (Batch Normalization)**: Этот метод помогает стабилизировать и ускорить процесс обучения, нормализуя входные данные для каждого слоя в каждом батче. Он уменьшает проблему колебаний градиентов и улучшает общую стабильность обучения.

2. **Метод ансамблей (Ensemble Methods)**: Создание нескольких моделей и комбинирование их предсказаний может помочь снизить риск переобучения. Ансамбли, такие как стекинг (stacking) или бейзийские сети, могут улучшить обобщающую способность модели.

3. **Ранняя остановка (Early Stopping)**: Это метод регуляризации, при котором обучение останавливается до достижения полного переобучения. Процесс обучения отслеживается с помощью метрики валидации, и как только она начинает ухудшаться, обучение прерывается.

4. **Dropout**: Dropout — это популярный метод, ко

In [24]:
from google.colab import files
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Выберите PDF-документ по тематике глубокого обучения")
uploaded = files.upload()

pdf_files = [
    file_name
    for file_name in uploaded
    if file_name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("PDF-файл не выбран")

pdf_path = pdf_files[0]

# Извлечение текста из PDF
documents = PyMuPDFLoader(pdf_path).load()

# Разбиение текста на фрагменты
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True
)

chunks = text_splitter.split_documents(documents)

print("\nФайл:", pdf_path)
print("Количество страниц:", len(documents))
print("Количество чанков:", len(chunks))

print("\nПример первого чанка:\n")
print(chunks[0].page_content[:1000])

Выберите PDF-документ по тематике глубокого обучения


Saving Тестовый материал для RAG.pdf to Тестовый материал для RAG.pdf

Файл: Тестовый материал для RAG.pdf
Количество страниц: 6
Количество чанков: 12

Пример первого чанка:

Практическая работа №12 · Тестовый PDF для RAG
Страница 1
Краткий справочник по глубокому
обучению
Тестовый PDF-документ для проверки Retrieval Augmented Generation (RAG)
Назначение документа. Этот материал создан специально для практической работы №12. Его
можно загрузить в интеллектуального агента, разбить на чанки и использовать для проверки
ответов на основе PDF-контекста.
1. Основные понятия
Глубокое обучение — направление машинного обучения, в котором используются нейронные
сети с несколькими слоями преобразований. Каждый слой формирует новое представление
входных данных. В процессе обучения параметры сети изменяются так, чтобы уменьшить
значение функции потерь.
Параметры модели — обучаемые числовые значения, например веса и смещения. Число
параметров зависит от архитектуры сети. Большая модель способна о

## Создание векторной базы данных

Для каждого чанка вычисляется эмбеддинг.  
Векторы сохраняются в ChromaDB и используются для семантического поиска.

In [25]:
import shutil
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

CHROMA_PATH = "./chroma_db_pr12"

# Удаление старой БД при повторном запуске ячейки
shutil.rmtree(CHROMA_PATH, ignore_errors=True)

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K}
)

print("Векторная база данных создана")
print("Количество добавленных чанков:", len(chunks))

Векторная база данных создана
Количество добавленных чанков: 12


In [27]:
def format_pdf_documents(documents):
    """
    Объединяет найденные чанки в единый текстовый контекст.
    """
    parts = []

    for index, document in enumerate(documents, start=1):
        page = document.metadata.get("page", "?")
        content = document.page_content

        parts.append(
            f"[Фрагмент {index}, страница {page}]\n{content}"
        )

    return "\n\n---\n\n".join(parts)


def answer_from_pdf(question):
    """
    Формирует ответ строго на основе загруженного PDF-документа.
    """
    relevant_documents = retriever.invoke(question)
    pdf_context = format_pdf_documents(relevant_documents)

    prompt = f"""
Ты — специализированный помощник по глубокому обучению.

Ответь на вопрос пользователя только на основе контекста из PDF-документа.
Если в контексте нет ответа, прямо напиши:
"В загруженном PDF-документе недостаточно информации для ответа."

Не добавляй сведения из собственных знаний.
Отвечай кратко и на грамотном русском языке.

КОНТЕКСТ ИЗ PDF:
{pdf_context}

ВОПРОС:
{question}

ОТВЕТ:
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0.1}
    )

    return response["message"]["content"]


# Проверка ответа по уникальной информации из PDF
question = "Какой контрольный маркер указан в тестовом документе?"

print("Вопрос:", question)
print("\nОтвет агента:")
print(answer_from_pdf(question))

Вопрос: Какой контрольный маркер указан в тестовом документе?

Ответ агента:
Контрольный маркер, указанный в тестовом документе, — "лазурный контур".


## Объединение PDF-RAG и веб-поиска

Итоговый агент использует два источника:
- загруженный PDF-документ;
- актуальные результаты веб-поиска Tavily.

В ответе отдельно указываются внешние веб-источники.

In [30]:
def answer_with_pdf_and_web(question):
    """
    Формирует ответ на основе PDF-документа и результатов Tavily.
    """
    # Поиск по PDF
    relevant_documents = retriever.invoke(question)
    pdf_context = format_pdf_documents(relevant_documents)

    # Поиск в интернете
    web_context, sources = search_web(question)

    prompt = f"""
Ты — специализированный интеллектуальный помощник по глубокому обучению.
Отвечай только на грамотном русском языке.

Правила:
1. Используй контекст из PDF и веб-поиска.
2. Если источники противоречат друг другу, сообщи об этом.
3. Не выдумывай сведения, которых нет в предоставленном контексте.
4. Дай понятный и структурированный ответ.
5. Не добавляй ссылки самостоятельно: они будут добавлены программой.

КОНТЕКСТ ИЗ PDF:
{pdf_context}

КОНТЕКСТ ИЗ ВЕБ-ПОИСКА:
{web_context}

ВОПРОС:
{question}

ОТВЕТ:
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0.1}
    )

    answer = response["message"]["content"]

    if sources:
        answer += "\n\nВеб-источники:\n" + "\n".join(sources)

    return answer


# Проверка объединенного агента
question = "Что такое RAG и какие этапы индексации PDF перечислены в документе?"

print("Вопрос:", question)
print("\nОтвет агента:\n")
print(answer_with_pdf_and_web(question))

Вопрос: Что такое RAG и какие этапы индексации PDF перечислены в документе?

Ответ агента:

**Что такое RAG?**

RAG (Retrieval Augmented Generation) — это метод, объединяющий генеративные языковые модели с системами поиска. Основная идея RAG заключается в том, чтобы улучшить ответы искусственного интеллекта, дополняя их информацией из внешних источников знаний, таких как базы данных или документы. Вместо того чтобы полагаться только на собственные внутренние знания, модель сначала ищет необходимую информацию во внешних источниках перед генерацией ответа.

**Этапы индексации PDF в RAG:**

В документе перечислены следующие этапы индексации PDF-документа для использования в системе RAG:

1. **Извлечение текста:** Извлекается текст из страниц PDF.
2. **Разбиение на чанки:** Текст разбивается на небольшие фрагменты (чанки) с небольшим перекрытием.
3. **Вычисление эмбеддингов:** Для каждого чанка вычисляется числовой вектор (эмбеддинг), представляющий его смысл.
4. **Хранение в векторной баз

## История диалога и сохранение результатов

Каждый вопрос и ответ записываются в историю.  
Пользователь может скачать оформленный текстовый файл с результатами беседы.

In [31]:
from datetime import datetime
from google.colab import files

chat_history = []


def ask_agent(question):
    """
    Отправляет вопрос объединенному агенту и сохраняет результат.
    """
    if not question or not question.strip():
        return "Введите непустой вопрос."

    answer = answer_with_pdf_and_web(question.strip())

    chat_history.append({
        "time": datetime.now().strftime("%d.%m.%Y %H:%M:%S"),
        "question": question.strip(),
        "answer": answer
    })

    return answer


def save_chat_history(file_name="История_диалога_ПР12.txt"):
    """
    Сохраняет историю беседы в TXT-файл и предлагает скачать его.
    """
    if not chat_history:
        print("История диалога пока пуста")
        return

    with open(file_name, "w", encoding="utf-8") as file:
        file.write("ИСТОРИЯ ВЗАИМОДЕЙСТВИЯ С LLM-АГЕНТОМ\n")
        file.write("Специализация: глубокое обучение\n")
        file.write("=" * 70 + "\n\n")

        for index, item in enumerate(chat_history, start=1):
            file.write(f"ЗАПРОС №{index}\n")
            file.write(f"Время: {item['time']}\n")
            file.write("-" * 70 + "\n")
            file.write(f"Пользователь:\n{item['question']}\n\n")
            file.write(f"Агент:\n{item['answer']}\n")
            file.write("=" * 70 + "\n\n")

    print("История сохранена:", file_name)
    files.download(file_name)

In [32]:
question = "Какой размер батча используется в эксперименте Маяк-12?"

print("Вопрос:", question)
print("\nОтвет агента:\n")
print(ask_agent(question))

print("\nКоличество записей в истории:", len(chat_history))

Вопрос: Какой размер батча используется в эксперименте Маяк-12?

Ответ агента:

Размер батча, используемый в эксперименте "Маяк-12", составляет **32**. Эта информация указана в разделе "Сводка параметров кейса" на странице 4 тестового PDF.

Веб-источники:
1. [PDF] АВТОМАТИЗАЦИЯ И ИЗМЕРЕНИЯ В МАШИНО
   https://www.sevsu.ru/upload/iblock/c88/xxgi101m3nehe2vx4mjvzipuc2pdalxx/2024_4(28).pdf
2. МАЯК-12-СТ - Оповещатель охранно-пожарный световой ...
   https://amadon.ru/product/mayak-12-st-opoveshhatel-ohranno-pozharnyj-svetovoj-stroboskopicheskij-netping
3. rockets .coffee | кофейная компания | Кофе из разных стран ...
   https://www.instagram.com/reel/DXwRhKiKH3Z

Количество записей в истории: 1


In [33]:
save_chat_history()

История сохранена: История_диалога_ПР12.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
import gradio as gr
import chromadb
import subprocess
import time
import ollama as _ollama
from datetime import datetime
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# ─────────────────────────────────────────────
# Вспомогательная функция - проверка Ollama
# ─────────────────────────────────────────────

def ensure_ollama():
    """Проверяет, запущен ли Ollama, и запускает при необходимости."""
    for _ in range(3):
        try:
            _ollama.list()
            return
        except Exception:
            subprocess.Popen(
                ["ollama", "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            time.sleep(4)

# ─────────────────────────────────────────────
# Обработка PDF
# ─────────────────────────────────────────────

def process_pdf_gradio(file):
    """
    Загружает PDF через Gradio и пересоздает глобальный retriever.
    """
    global retriever

    if file is None:
        return "Файл не выбран"

    ensure_ollama()

    try:
        documents = PyMuPDFLoader(file.name).load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            add_start_index=True
        )
        chunks = splitter.split_documents(documents)

        # EphemeralClient хранит всё в памяти — никаких прав на диск не нужно
        client = chromadb.EphemeralClient()

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=OllamaEmbeddings(model=EMBEDDING_MODEL),
            client=client,
            collection_name="gradio_pdf"
        )

        retriever = vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": RETRIEVER_K}
        )

        return f"Загружено: {len(documents)} стр., {len(chunks)} чанков"

    except Exception as e:
        return f"Ошибка: {e}"


# ─────────────────────────────────────────────
# Чат
# ─────────────────────────────────────────────

def chat_fn(question, history, mode):
    if not question.strip():
        return history, ""

    if mode == "PDF + веб (комбинированный)":
        answer = ask_agent(question)
    elif mode == "Только веб (Tavily)":
        answer = answer_with_web_search(question)
        chat_history.append({
            "time": datetime.now().strftime("%d.%m.%Y %H:%M:%S"),
            "question": question.strip(),
            "answer": answer
        })
    else:
        answer = answer_from_pdf(question)
        chat_history.append({
            "time": datetime.now().strftime("%d.%m.%Y %H:%M:%S"),
            "question": question.strip(),
            "answer": answer
        })

    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    return history, ""


def save_fn():
    if not chat_history:
        return "История диалога пуста"

    file_name = "История_диалога_ПР12.txt"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write("ИСТОРИЯ ВЗАИМОДЕЙСТВИЯ С LLM-АГЕНТОМ\n")
        f.write("Специализация: глубокое обучение\n")
        f.write("=" * 70 + "\n\n")
        for index, item in enumerate(chat_history, start=1):
            f.write(f"ЗАПРОС №{index}\n")
            f.write(f"Время: {item['time']}\n")
            f.write("-" * 70 + "\n")
            f.write(f"Пользователь:\n{item['question']}\n\n")
            f.write(f"Агент:\n{item['answer']}\n")
            f.write("=" * 70 + "\n\n")

    return f"Сохранено: {file_name}  ({len(chat_history)} записей)"


# ─────────────────────────────────────────────
# Интерфейс
# ─────────────────────────────────────────────

with gr.Blocks(title="LLM-агент по глубокому обучению") as demo:

    gr.Markdown(
        "# LLM-агент по глубокому обучению\n"
        "**Практическая работа №12** · модель `aya-expanse:8b` + Tavily + PDF-RAG"
    )

    with gr.Row():
        pdf_upload = gr.File(
            label="Загрузить PDF",
            file_types=[".pdf"],
            scale=3
        )
        pdf_status = gr.Textbox(
            label="Статус PDF",
            interactive=False,
            scale=2
        )

    mode = gr.Radio(
        choices=[
            "PDF + веб (комбинированный)",
            "Только веб (Tavily)",
            "Только PDF",
        ],
        value="PDF + веб (комбинированный)",
        label="Режим работы агента",
    )

    chatbot = gr.Chatbot(label="Диалог", height=400)

    with gr.Row():
        question_input = gr.Textbox(
            placeholder="Введите вопрос по глубокому обучению...",
            label="Вопрос",
            lines=2,
            scale=5,
        )
        submit_btn = gr.Button("Отправить", variant="primary", scale=1)

    with gr.Row():
        clear_btn = gr.Button("Очистить чат", variant="secondary")
        save_btn  = gr.Button("Сохранить историю", variant="secondary")

    status_box = gr.Textbox(label="Статус", interactive=False, lines=1)

    pdf_upload.change(
        fn=process_pdf_gradio,
        inputs=pdf_upload,
        outputs=pdf_status
    )

    submit_btn.click(
        fn=chat_fn,
        inputs=[question_input, chatbot, mode],
        outputs=[chatbot, question_input],
    )

    question_input.submit(
        fn=chat_fn,
        inputs=[question_input, chatbot, mode],
        outputs=[chatbot, question_input],
    )

    clear_btn.click(
        fn=lambda: ([], ""),
        outputs=[chatbot, question_input],
    )

    save_btn.click(
        fn=save_fn,
        outputs=status_box,
    )

demo.launch(share=True, debug=False, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://12601251b7b38eb232.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [60]:
# Сохранение моделей Ollama на Google Drive (один раз)
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

src = os.path.expanduser("~/.ollama")
dst = "/content/drive/MyDrive/ollama_models"

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Модели сохранены на Google Drive")
else:
    print("Модели уже есть на Drive")

Mounted at /content/drive
Модели сохранены на Google Drive
